## 1. Imports and Setup

In [1]:
# Cell 1: Imports and configuration
import pandas as pd
import re
import json
from pathlib import Path
from typing import Optional, List, Dict
import ollama
import time

# Configuration
DATA_DIR = Path("data")
MODEL_NAME = "qwen3.5:0.8b"  # Adjust to your downloaded model name
MAX_CONTEXT_LENGTH = 4000  # Truncate very long contexts
TEMPERATURE = 0.1  # Low temperature for more deterministic answers
TIMEOUT_SECONDS = 60  # Timeout for LLM requests

# Set pandas options for better display
pd.set_option('display.max_colwidth', None)

## 2. Text Cleaning Functions

In [2]:
# Cell 2: Text preprocessing functions
def clean_context(text: str) -> str:
    """
    Clean contract text by removing HTML tags, excessive whitespace, 
    and other artifacts while preserving meaningful content.
    
    Args:
        text: Raw context text from the dataset
        
    Returns:
        Cleaned text suitable for LLM processing
    """
    if not isinstance(text, str):
        return ""
    
    # Remove HTML/XML tags
    text = re.sub(r'<[^>]+>', '', text)
    
    # Remove markdown-style links but keep the text: [text](url) -> text
    text = re.sub(r'\[([^\]]+)\]\([^)]+\)', r'\1', text)
    
    # Remove excessive whitespace (multiple spaces/tabs/newlines)
    text = re.sub(r'\s+', ' ', text)
    
    # Remove control characters except basic whitespace
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)
    
    # Normalize quotes and dashes
    text = text.replace('""', '"').replace("''", "'")
    text = text.replace('—', '-').replace('–', '-')
    
    # Strip leading/trailing whitespace
    text = text.strip()
    
    return text


def truncate_context(text: str, max_length: int = MAX_CONTEXT_LENGTH) -> str:
    """
    Truncate context to fit within model's context window while 
    trying to preserve the most relevant content.
    
    Args:
        text: Cleaned context text
        max_length: Maximum number of characters to keep
        
    Returns:
        Truncated text
    """
    if len(text) <= max_length:
        return text
    
    # Keep the beginning and end, as contract answers often appear in definitions or clauses
    # Keep ~60% from start, ~40% from end
    start_len = int(max_length * 0.6)
    end_len = max_length - start_len
    
    return text[:start_len] + " ... [truncated] ... " + text[-end_len:]

## 3. Data Loading and Preparation

In [3]:
# Cell 3: Load and prepare datasets
def load_data(data_dir: Path = DATA_DIR) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Load train and test datasets.
    
    Args:
        data_dir: Path to data directory
        
    Returns:
        Tuple of (train_df, test_df)
    """
    train_path = data_dir / "train.csv"
    test_path = data_dir / "test.csv"
    
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    
    print(f"Train shape: {train_df.shape}")
    print(f"Test shape: {test_df.shape}")
    
    return train_df, test_df


def prepare_dataframe(df: pd.DataFrame, is_train: bool = True) -> pd.DataFrame:
    """
    Apply preprocessing to dataframe: clean context, parse answers.
    
    Args:
        df: Input dataframe
        is_train: Whether this is training data (has 'answers' column)
        
    Returns:
        Preprocessed dataframe
    """
    df = df.copy()
    
    # Clean context column
    df['context_clean'] = df['context'].apply(clean_context)
    df['context_clean'] = df['context_clean'].apply(truncate_context)
    
    # Parse answers column (it's stored as a string representation of a list)
    if is_train and 'answers' in df.columns:
        def parse_answers(ans_str: str) -> str:
            """Extract the primary answer from the answers field."""
            if pd.isna(ans_str) or ans_str == '':
                return ""
            try:
                # Handle string representation of list
                if isinstance(ans_str, str) and ans_str.startswith('['):
                    answers_list = json.loads(ans_str.replace("'", '"'))
                    return answers_list[0] if answers_list else ""
                return str(ans_str)
            except:
                return str(ans_str)
        
        df['answer_target'] = df['answers'].apply(parse_answers)
    
    return df

## 4. LLM Prompt Engineering and Inference

In [10]:
# Cell 4: LLM inference with prompt engineering
def build_prompt(question: str, context: str) -> str:
    """
    Construct a prompt for extractive question answering.
    
    Args:
        question: The question to answer
        context: The contract text context
        
    Returns:
        Formatted prompt string
    """
    prompt = f"""You are a contract analysis assistant. Answer the question based ONLY on the provided contract text.

                Question: {question}
                
                Contract Text:
                {context}
                
                Instructions:
                1. Find the exact answer in the contract text above
                2. Return ONLY the answer text, nothing else
                3. If the answer is a date, return it in the format shown in the text
                4. If the answer is a party name, return it exactly as written
                5. If you cannot find the answer in the text, respond with: "Not found in document"
                
                Answer:"""
    return prompt


def get_llm_answer(question: str, context: str, model: str = MODEL_NAME) -> Optional[str]:
    """
    Query the local LLM for an answer.
    
    Args:
        question: Question to answer
        context: Context text
        model: Ollama model name
        
    Returns:
        Extracted answer string or None if error
    """
    prompt = build_prompt(question, context)
    
    try:
        response = ollama.chat(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            options={
                "temperature": TEMPERATURE,
                "num_predict": 150  # Limit response length
            },
            # timeout=TIMEOUT_SECONDS
        )
        
        answer = response['message']['content'].strip()
        
        # Post-process: remove common LLM artifacts
        answer = re.sub(r'^Answer:\s*', '', answer, flags=re.IGNORECASE)
        answer = re.sub(r'^["\']|["\']$', '', answer)  # Remove surrounding quotes
        answer = answer.strip()
        
        return answer if answer else None
        
    except Exception as e:
        print(f"Error querying LLM: {e}")
        return None


def batch_predict(df: pd.DataFrame, batch_size: int = 10) -> pd.DataFrame:
    """
    Generate predictions for a dataframe using the LLM.
    
    Args:
        df: DataFrame with 'question' and 'context_clean' columns
        batch_size: Process in batches to manage memory/time
        
    Returns:
        DataFrame with added 'prediction' column
    """
    df = df.copy()
    df['prediction'] = None
    
    for idx, row in df.iterrows():
        question = row['question']
        context = row['context_clean']
        
        prediction = get_llm_answer(question, context)
        df.at[idx, 'prediction'] = prediction
        
        # Progress indicator
        if (idx + 1) % batch_size == 0:
            print(f"Processed {idx + 1}/{len(df)} rows")
            time.sleep(1)  # Small delay to avoid overwhelming the model
    
    return df

## 5. Evaluation Function (for validation)

In [11]:
# Cell 5: Exact match evaluation
def calculate_exact_match(predictions: pd.Series, targets: pd.Series) -> float:
    """
    Calculate exact match accuracy.
    
    Args:
        predictions: Series of predicted answers
        targets: Series of ground truth answers
        
    Returns:
        Exact match score (0.0 to 1.0)
    """
    # Normalize both for comparison
    def normalize(text: str) -> str:
        if pd.isna(text):
            return ""
        text = str(text).lower().strip()
        # Remove extra whitespace
        text = re.sub(r'\s+', ' ', text)
        return text
    
    matches = sum(
        normalize(pred) == normalize(target) 
        for pred, target in zip(predictions, targets)
    )
    
    return matches / len(targets) if len(targets) > 0 else 0.0


def validate_on_train(train_df: pd.DataFrame, sample_size: int = 50) -> float:
    """
    Quick validation on a sample of training data.
    
    Args:
        train_df: Preprocessed training dataframe
        sample_size: Number of samples to evaluate
        
    Returns:
        Exact match score on sample
    """
    sample = train_df.sample(n=min(sample_size, len(train_df)), random_state=42)
    sample = batch_predict(sample)
    
    em_score = calculate_exact_match(
        sample['prediction'], 
        sample['answer_target']
    )
    
    print(f"Sample Exact Match: {em_score:.3f} ({sample.shape[0]} samples)")
    return em_score

## 6. Main Execution Pipeline

In [12]:
# Cell 6: Main pipeline - run this last
def main():
    """Main execution function."""
    print("=== Loading Data ===")
    train_df, test_df = load_data()
    
    print("\n=== Preprocessing ===")
    train_df = prepare_dataframe(train_df, is_train=True)
    test_df = prepare_dataframe(test_df, is_train=False)
    
    # Optional: Quick validation on training sample
    print("\n=== Quick Validation (optional) ===")
    # Uncomment to run validation (takes time):
    validate_on_train(train_df, sample_size=20)
    
    # print("\n=== Generating Predictions ===")
    # test_df = batch_predict(test_df)
    
    # print("\n=== Creating Submission ===")
    # submission = test_df[['id', 'prediction']].copy()
    # submission.columns = ['id', 'answers']  # Match expected format
    
    # # Save submission
    # output_path = DATA_DIR / "submission.csv"
    # submission.to_csv(output_path, index=False)
    # print(f"Submission saved to: {output_path}")
    
    # # Show sample predictions
    # print("\n=== Sample Predictions ===")
    # sample_preds = test_df[['question', 'answer_target', 'prediction']].head(10)
    # print(sample_preds.to_markdown(index=False))
    
    return submission

In [13]:
# Run the pipeline
if __name__ == "__main__":
    submission_df = main()

=== Loading Data ===
Train shape: (200, 5)
Test shape: (165, 3)

=== Preprocessing ===

=== Quick Validation (optional) ===
Processed 70/20 rows
Sample Exact Match: 0.000 (20 samples)


NameError: name 'submission' is not defined